<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-04-rag/lesson-4.1-document-ai/notebooks/GCP_Capstone_4.1_Document_AI.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.1 Document AI — OCR, Layout Parser, Form Parser
**Netsetos GenAI Engineering — GCP Capstone**

Convert PDFs, scans, and forms into clean text for RAG.


## Setup

The Firestore `(default)` database is created below in **asia-south1 (Mumbai)**, and that location is permanent once chosen. Document AI processors live in the `us` location. Data at rest stays in India; OCR traffic goes to the processor location.

**The documents are real.** Two of the thirteen real documents in DocuMind's corpus (`deploy/evals/real_sources.json`) do this lesson's work: the **POSH Act, 2013** as a *scanned* Gazette of India — JBIG2 page images, no text layer, the kind of file a tenant actually uploads — and the **Payment of Gratuity Act, 1972** as the ministry's text PDF. The cell takes them from the course repo when it is beside this notebook (or clones it the way 10.4 does) and otherwise straight from the publisher, checking the sha256 the kit recorded. The onboarding **form** stays synthetic: a filled form is personal data by definition, and none of that goes on a shared screen.


In [ ]:
!pip install -q google-cloud-documentai google-cloud-firestore==2.30.0 google-genai==2.21.0 reportlab pypdf
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE
LOCATION = 'us'  # Document AI location: 'us' or 'eu'
TENANT_ID = 'acme'  # the teaching tenant; every chunk this lesson writes carries it

# --- 0. APIs: 1.1 enabled these for the capstone project; re-running is idempotent ---
!gcloud services enable documentai.googleapis.com aiplatform.googleapis.com firestore.googleapis.com --project {PROJECT_ID}

# --- 1. The real documents (deploy/evals/real_sources.json): kit first, publisher second ---
import hashlib, os, subprocess, urllib.request
REAL = {   # slug: (where the ministry / state government publishes it, sha256 of the bytes the kit recorded)
    'posh_act_2013': ('https://gil.gujarat.gov.in/Media/DocumentUpload/posh_act._english.pdf',
                      '70d525e419baf7a9d249d9c679a23714603509d018d38f932be3a18365d3b195'),      # 13 pages, a SCANNED Gazette: no text layer
    'payment_of_gratuity_act_1972': ('https://www.labour.gov.in/static/uploads/2025/06/072a4b7ea8246533c62b96b68a30da53.pdf',
                                     'fa044c91c6714c738ded7ba98dbe9950060d9bf85ec2a63996cee927269669d4'),      # 10 pages, text PDF
}

def real_document(slug):
    """The PDF from the kit beside this notebook (or its clone under /content); else from the publisher."""
    for base in ('deploy/evals/corpus/acme', '../deploy/evals/corpus/acme',
                 '/content/agentic-ai-weekend-gcp/deploy/evals/corpus/acme'):
        if os.path.isfile(f'{base}/{slug}.pdf'):
            return f'{base}/{slug}.pdf'
    if not os.path.isdir('/content/agentic-ai-weekend-gcp'):
        subprocess.run(['git', 'clone', '--depth', '1', '-b', 'feat/lesson-4.8-live-evals', 'https://github.com/netsetos/agentic-ai-weekend-gcp',
                        '/content/agentic-ai-weekend-gcp'], check=False)
    if os.path.isfile(f'/content/agentic-ai-weekend-gcp/deploy/evals/corpus/acme/{slug}.pdf'):
        return f'/content/agentic-ai-weekend-gcp/deploy/evals/corpus/acme/{slug}.pdf'
    url, sha = REAL[slug]
    data = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'}),
                                  timeout=180).read()
    if hashlib.sha256(data).hexdigest() != sha:
        print(f'  {slug}: the publisher changed this file since the kit recorded it - read it before trusting it')
    with open(f'{slug}.pdf', 'wb') as f:
        f.write(data)
    return f'{slug}.pdf'

POSH_PDF = real_document('posh_act_2013')                     # Cell 2: OCR on a scan
GRATUITY_PDF = real_document('payment_of_gratuity_act_1972')  # Cells 3, 5, 6: layout, chunks, ingest
print(f'Real documents: {POSH_PDF} ({os.path.getsize(POSH_PDF):,} bytes), '
      f'{GRATUITY_PDF} ({os.path.getsize(GRATUITY_PDF):,} bytes)')

# --- 1b. A synthetic form.pdf (labelled fields + a table) for the Form Parser (Cell 4). A filled
#         form is personal data by definition, so this one is drawn, not downloaded. ---
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
_f = canvas.Canvas('form.pdf', pagesize=letter)
_f.setFont('Helvetica-Bold', 15); _f.drawString(72, 730, 'DocuMind AI - Onboarding Form')
_f.setFont('Helvetica', 11)
for _i, _line in enumerate([
        'Full Name: Priya Sharma', 'Employee ID: EMP-2026-118',
        'Department: Machine Learning', 'Date of Joining: 2026-09-01',
        'Location: Hyderabad', 'Email: priya.sharma@acme.in',
        'Manager: Rahul Verma']):
    _f.drawString(72, 700 - _i * 24, _line)
_f.setFont('Helvetica-Bold', 11); _f.drawString(72, 510, 'Equipment Issued')
_y = 490
for _r, (_a, _b) in enumerate([('Item', 'Serial'), ('Laptop', 'LT-9921'),
                               ('Monitor', 'MN-4415'), ('Access Card', 'AC-7788')]):
    _f.setFont('Helvetica-Bold' if _r == 0 else 'Helvetica', 10)
    _f.drawString(80, _y, _a); _f.drawString(260, _y, _b)
    _f.line(72, _y - 5, 400, _y - 5)
    _y -= 22
_f.showPage(); _f.save()
print('Wrote form.pdf')

# --- 2. Document AI processors: created once if missing (or paste your own IDs) ---
from google.api_core.client_options import ClientOptions
from google.cloud import documentai
_da = documentai.DocumentProcessorServiceClient(
    client_options=ClientOptions(api_endpoint=f'{LOCATION}-documentai.googleapis.com'))
_parent = _da.common_location_path(PROJECT_ID, LOCATION)

def _ensure_processor(display_name, type_):
    # reuse an existing processor of this type, else create one (needs the Document
    # AI API enabled + documentai.processors.create; else create in the Console and
    # paste the ID here).
    for p in _da.list_processors(parent=_parent):
        if p.type_ == type_:
            return p.name.split('/')[-1]
    p = _da.create_processor(parent=_parent,
        processor=documentai.Processor(type_=type_, display_name=display_name))
    return p.name.split('/')[-1]

OCR_ID    = _ensure_processor('documind-ocr',    'OCR_PROCESSOR')
LAYOUT_ID = _ensure_processor('documind-layout', 'LAYOUT_PARSER_PROCESSOR')
FORM_ID   = _ensure_processor('documind-form',   'FORM_PARSER_PROCESSOR')
print(f'Processors ready: OCR={OCR_ID}  LAYOUT={LAYOUT_ID}  FORM={FORM_ID}')

# --- 3. Firestore (default) database for Cell 5 (created once if missing) ---
import subprocess
FIRESTORE_LOCATION = 'asia-south1'  # region for the (default) DB; PERMANENT once created
if '(default)' not in subprocess.run(
        ['gcloud', 'firestore', 'databases', 'list', '--project', PROJECT_ID, '--format=value(name)'],
        capture_output=True, text=True).stdout:
    print(f'Creating Firestore (default) database in {FIRESTORE_LOCATION} (one-time)...')
    subprocess.run(['gcloud', 'firestore', 'databases', 'create',
                    '--location=' + FIRESTORE_LOCATION, '--project', PROJECT_ID], check=False)
else:
    print('Firestore (default) database ready.')

## Cell 1: Universal Processing Function


In [ ]:
from google.api_core.client_options import ClientOptions
from google.cloud import documentai

def process_document(project_id, location, processor_id, file_path,
                     mime_type='application/pdf', process_options=None):
    client = documentai.DocumentProcessorServiceClient(
        client_options=ClientOptions(
            api_endpoint=f'{location}-documentai.googleapis.com'))
    name = client.processor_path(project_id, location, processor_id)
    with open(file_path, 'rb') as f:
        content = f.read()
    request = documentai.ProcessRequest(
        name=name,
        raw_document=documentai.RawDocument(content=content, mime_type=mime_type),
        process_options=process_options)
    return client.process_document(request=request).document

print('process_document() ready')


## Cell 2: OCR — Full Text Extraction

The POSH Act as the Gazette of India printed it: a scan. Five of its thirteen pages carry a text layer, and it is garbage (the cell shows you page 3); the other eight have none. That is exactly why an OCR lane exists - and why native PDF parsing is switched off here: it would trust that layer. 13 pages fits the 15-page cap on online (synchronous) requests; a longer scan goes through the batch pattern on the lesson page. About $0.02 of OCR.


In [ ]:
# The scanned POSH Act: 13 page images. Five of them carry a text layer, and it is garbage - what a
# PDF library sees on page 3 below; from page 6 on there is none at all (pypdf returns '').
from pypdf import PdfReader
_pypdf = (PdfReader(POSH_PDF).pages[2].extract_text() or '')
print(f'pypdf, page 3: {len(_pypdf)} chars, e.g. {_pypdf[:80]!r}')

options = documentai.ProcessOptions(
    ocr_config=documentai.OcrConfig(
        enable_native_pdf_parsing=False,      # OCR every page: the text layer five pages carry is garbled, and native parsing would trust it
        enable_image_quality_scores=True,
        hints=documentai.OcrConfig.Hints(language_hints=['en','hi'])))   # the Gazette prints both

doc = process_document(PROJECT_ID, LOCATION, OCR_ID, POSH_PDF,
                       process_options=options)

print(f'\nDocument AI OCR: {len(doc.text)} chars from {len(doc.pages)} pages')
print(doc.text[:500])
def _anchor_text(document, text_anchor):
    # Online process() leaves text_anchor.content empty; resolve text from document.text
    if not text_anchor.text_segments:
        return (text_anchor.content or '').strip()
    return ''.join(document.text[int(s.start_index):int(s.end_index)]
                   for s in text_anchor.text_segments).strip()

for page in doc.pages[:4]:
    langs = [(l.language_code, f'{l.confidence:.0%}') for l in page.detected_languages]
    q = page.image_quality_scores.quality_score if page.image_quality_scores else None
    print(f'  Page {page.page_number}: {len(page.paragraphs)} paragraphs, langs={langs}, '
          f'quality={q:.2f}' if q is not None else
          f'  Page {page.page_number}: {len(page.paragraphs)} paragraphs, langs={langs}')


## Cell 3: Layout Parser — RAG-Ready Chunks

The Payment of Gratuity Act, 1972 (10 pages, text PDF). Section headings become ancestor headings on every chunk, and each chunk carries the page span it came from — the `page_start` a citation names in 4.2 and 4.5.


In [ ]:
options = documentai.ProcessOptions(
    layout_config=documentai.ProcessOptions.LayoutConfig(
        enable_table_annotation=True,
        enable_image_annotation=True,
        chunking_config=documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
            chunk_size=1024,
            include_ancestor_headings=True)))

doc = process_document(PROJECT_ID, LOCATION, LAYOUT_ID, GRATUITY_PDF,
                       process_options=options)

print(f'Chunks: {len(doc.chunked_document.chunks)} from {GRATUITY_PDF}')
for i, chunk in enumerate(doc.chunked_document.chunks[:5]):
    print(f'\n--- Chunk {i} ({chunk.chunk_id}) ---')
    print(f'Pages: {chunk.page_span.page_start}-{chunk.page_span.page_end}')
    print(f'Content:\n{chunk.content[:200]}...')


## Cell 4: Form Parser — Key-Value Pairs


In [ ]:
doc = process_document(PROJECT_ID, LOCATION, FORM_ID, 'form.pdf')

for page in doc.pages:
    print(f'\n=== Page {page.page_number} ===')
    for field in page.form_fields:
        key = _anchor_text(doc, field.field_name.text_anchor)
        val = _anchor_text(doc, field.field_value.text_anchor)
        print(f'  {key}: {val} ({field.field_value.confidence:.0%})')
    for idx, table in enumerate(page.tables):
        print(f'\n  Table {idx}:')
        for row in table.body_rows:
            cells = [_anchor_text(doc, c.layout.text_anchor) for c in row.cells]
            print(f'    {cells}')


## Cell 5: Store Chunks in Firestore

Into `chunks` — DocuMind's ONE collection — as the canonical document (`tenant_id`, `text`, `source_uri`, `page_start`, `doc_type`, `embedding`, `kind`), which is what 4.2's `find_nearest` and the 12.5 ingest worker read and write. The chunk id is `acme:payment_of_gratuity_act_1972#d3`: tenant, document slug, Document AI chunk number — so 4.7's `must_retrieve` anchors match it, and a re-run overwrites instead of duplicating.


In [ ]:
import os
from google import genai
from google.genai import types
from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector

EMBED_MODEL = 'text-embedding-005'
EMBED_CONFIG = types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT', output_dimensionality=768)
emb = genai.Client(enterprise=True, project=PROJECT_ID, location='us-central1')  # embeddings are regional-only
CHUNKS_COLLECTION = 'chunks'   # one name, from 2.3 to production (deploy/services/rag-api/config.py)

EMBED_BATCH, EMBED_TOKENS, CHARS_PER_TOKEN = 250, 15_000, 3


def embed_batches(texts: list) -> list:
    """Batches of at most EMBED_BATCH texts AND about EMBED_TOKENS tokens. text-embedding-005 takes
    250 texts per request and 20,000 tokens across them, and a request over either limit fails
    whole; a two-thousand-character chunk is ~500 tokens, so 250 of them are ~125,000. The first
    live corpus load (6 Sept 2026) failed every long Act exactly here - the same rule now lives in
    services/ingest/indexer.py and deploy/shared/documind_corpus.py."""
    out, cur, cur_tokens = [], [], 0
    for t in texts:
        tokens = max(1, len(t) // CHARS_PER_TOKEN)
        if cur and (len(cur) >= EMBED_BATCH or cur_tokens + tokens > EMBED_TOKENS):
            out.append(cur); cur, cur_tokens = [], 0
        cur.append(t); cur_tokens += tokens
    if cur:
        out.append(cur)
    return out

def embed_texts(texts):
    """Embed chunk contents, batched by count and by tokens (embed_batches above)."""
    vectors = []
    for batch in embed_batches(texts):
        resp = emb.models.embed_content(model=EMBED_MODEL, contents=batch, config=EMBED_CONFIG)
        vectors.extend(e.values for e in resp.embeddings)
    return vectors

def store_chunks(document, file_path, tenant_id=TENANT_ID, doc_type='statute', page_offset=0):
    """Embed Layout Parser chunks and write the canonical document into chunks (the 4.2 contract)."""
    db = firestore.Client(project=PROJECT_ID)
    slug = os.path.basename(file_path).rsplit('.', 1)[0]
    source_uri = f'gs://{PROJECT_ID}-uploads/{tenant_id}/{os.path.basename(file_path)}'  # where upload.sh puts it
    chunks = list(document.chunked_document.chunks) if document.chunked_document else []
    if not chunks:
        raise ValueError('no chunks on this document: parse it with the Layout Parser and a chunking_config (Cell 3), not the OCR processor')
    vectors = embed_texts([c.content for c in chunks])
    batch, pending = db.batch(), 0

    for i, (chunk, vec) in enumerate(zip(chunks, vectors)):
        ref = db.collection(CHUNKS_COLLECTION).document(f'{tenant_id}:{slug}#d{i}')
        batch.set(ref, {
            'tenant_id': tenant_id,                       # the field every query filters on
            'text': chunk.content,
            'embedding': Vector(vec),                     # 768-d: what 4.2 find_nearest queries
            'source_uri': source_uri,
            'page_start': chunk.page_span.page_start + page_offset,
            'page_end': chunk.page_span.page_end + page_offset,
            'doc_type': doc_type,
            'kind': 'text',
            'section': None,                              # the Layout Parser folds headings into the text
            'processed_at': firestore.SERVER_TIMESTAMP,
        })
        pending += 1
        if pending == 400:   # Firestore batches hold at most 500 ops: commit early
            batch.commit()
            batch, pending = db.batch(), 0
    if pending:
        batch.commit()
    print(f'Stored {len(chunks)} embedded chunks from {file_path} as tenant {tenant_id!r}')

# store_chunks(doc, GRATUITY_PDF)  # uncomment; 'doc' must be a Layout-parsed doc (re-run Cell 3)


## Cell 6: Document Ingestion Module

This is the hand-off to 4.2: `ask_documind()` runs `find_nearest` on the `embedding` field these writes create, and 4.5 reranks and packs the same chunks. The tenant-filtered vector index on `chunks` (`tenant_id` + `embedding`, 768-d) is created in 4.2's Setup and must exist before the first query.

Two production details live here. Online Document AI requests stop at **15 pages**, so the ingester slices a longer PDF with `pypdf` and processes each slice with a page offset (the 29-page Code on Wages becomes two calls). And a document the tenant already holds is **skipped**, on the same rule 4.2's loader applies — `tenant_id` + `source_uri` — so running the two lessons in either order leaves one copy of each Act.


In [ ]:
import os
from google import genai
from google.genai import types
from google.api_core.client_options import ClientOptions
from google.cloud import documentai, firestore
from google.cloud.firestore_v1.base_query import FieldFilter
from google.cloud.firestore_v1.vector import Vector
from pypdf import PdfReader, PdfWriter

EMBED_MODEL = 'text-embedding-005'
EMBED_CONFIG = types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT', output_dimensionality=768)
ONLINE_PAGE_LIMIT = 15   # Document AI online (synchronous) requests; batch takes 500 (OCR, Layout)

class DocumentIngester:
    def __init__(self, project_id, location='us', tenant_id=TENANT_ID):
        self.project_id = project_id
        self.location = location
        self.tenant_id = tenant_id
        self.client = documentai.DocumentProcessorServiceClient(
            client_options=ClientOptions(
                api_endpoint=f'{location}-documentai.googleapis.com'))
        self.db = firestore.Client(project=project_id)
        self.emb = genai.Client(enterprise=True, project=project_id, location='us-central1')  # embeddings are regional-only

    def process(self, processor_id, content, mime_type='application/pdf', process_options=None):
        name = self.client.processor_path(self.project_id, self.location, processor_id)
        request = documentai.ProcessRequest(
            name=name,
            raw_document=documentai.RawDocument(content=content, mime_type=mime_type),
            process_options=process_options)
        return self.client.process_document(request=request).document

    def embed_texts(self, texts):
        """Embed chunk contents, batched by count and by tokens (embed_batches, defined above)."""
        vectors = []
        for batch in embed_batches(texts):
            resp = self.emb.models.embed_content(model=EMBED_MODEL, contents=batch, config=EMBED_CONFIG)
            vectors.extend(e.values for e in resp.embeddings)
        return vectors

    @staticmethod
    def slices(file_path, limit=ONLINE_PAGE_LIMIT):
        """(pdf bytes, first page number) per slice of at most `limit` pages."""
        reader = PdfReader(file_path)
        for first in range(0, len(reader.pages), limit):
            w = PdfWriter()
            for p in reader.pages[first:first + limit]:
                w.add_page(p)
            import io
            buf = io.BytesIO(); w.write(buf)
            yield buf.getvalue(), first

    def already_ingested(self, source_uri):
        """The rule 4.2's loader applies too: one tenant, one source_uri, one set of chunks."""
        return bool(self.db.collection(CHUNKS_COLLECTION)
                    .where(filter=FieldFilter('tenant_id', '==', self.tenant_id))
                    .where(filter=FieldFilter('source_uri', '==', source_uri)).limit(1).get())

    def ingest_for_rag(self, layout_id, file_path, doc_type='statute', chunk_size=1024):
        """Full pipeline: slice -> Layout Parse -> Chunk -> Embed -> Store in chunks (the 4.2 contract)."""
        slug = os.path.basename(file_path).rsplit('.', 1)[0]
        source_uri = f'gs://{self.project_id}-uploads/{self.tenant_id}/{os.path.basename(file_path)}'
        if self.already_ingested(source_uri):
            print(f'{slug}: already in chunks for tenant {self.tenant_id!r} - skipped')
            return 0
        options = documentai.ProcessOptions(
            layout_config=documentai.ProcessOptions.LayoutConfig(
                chunking_config=documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
                    chunk_size=chunk_size,
                    include_ancestor_headings=True)))
        rows = []
        for content, page_offset in self.slices(file_path):
            doc = self.process(layout_id, content, process_options=options)
            for chunk in doc.chunked_document.chunks:
                rows.append({'text': chunk.content,
                             'page_start': chunk.page_span.page_start + page_offset,
                             'page_end': chunk.page_span.page_end + page_offset})
        vectors = self.embed_texts([r['text'] for r in rows])
        batch, pending = self.db.batch(), 0
        for i, (r, vec) in enumerate(zip(rows, vectors)):
            ref = self.db.collection(CHUNKS_COLLECTION).document(f'{self.tenant_id}:{slug}#d{i}')
            batch.set(ref, {
                'tenant_id': self.tenant_id, 'text': r['text'], 'embedding': Vector(vec),
                'source_uri': source_uri, 'page_start': r['page_start'], 'page_end': r['page_end'],
                'doc_type': doc_type, 'kind': 'text', 'section': None,
                'processed_at': firestore.SERVER_TIMESTAMP})
            pending += 1
            if pending == 400:   # Firestore batches hold at most 500 ops: commit early
                batch.commit()
                batch, pending = self.db.batch(), 0
        if pending:
            batch.commit()
        return len(rows)

print('DocumentIngester ready')

# Put the Gratuity Act into chunks so lesson 4.2 has Document AI chunks to query. This is a
# SECOND billed Layout Parser call on the file (Cell 3 already parsed it) plus one embedding
# request: a few cents for ten pages. 4.2 seeds the rest of the corpus from the kit's text
# mirrors and skips this Act because it is already here.
ingester = DocumentIngester(PROJECT_ID, LOCATION)
n = ingester.ingest_for_rag(LAYOUT_ID, GRATUITY_PDF)
print(f'Ingested {n} embedded chunks of {GRATUITY_PDF} into chunks as tenant {TENANT_ID!r} (4.2 queries these)' if n else
      f'{GRATUITY_PDF}: already in chunks for tenant {TENANT_ID!r} - nothing written, and that is the idempotent path working (4.2 queries the existing chunks)')


## ✅ Lesson 4.1 Complete!

- ✅ Universal processing pattern (ProcessRequest + RawDocument)
- ✅ Enterprise OCR on a real scanned Gazette (the POSH Act, 2013) — with the pypdf contrast that shows why OCR exists
- ✅ Layout Parser with include_ancestor_headings on a real Act (Payment of Gratuity, 1972), page spans on every chunk
- ✅ Form Parser with key-value pairs and tables (on a synthetic form — a filled form is personal data)
- ✅ Batch processing pattern (see the lesson page - GCS in/out URIs) and the 15-page online cap, sliced with pypdf
- ✅ The canonical document in `chunks`: tenant_id, text, source_uri, page_start, doc_type, embedding
- ✅ DocumentIngester production module, idempotent per tenant + source_uri

Run this notebook before 4.2 and the Gratuity Act is already in `chunks`; 4.2's loader seeds the rest of DocuMind's corpus (the other five Acts and the tenant's own documents) and skips what is here.

**Next: Lesson 4.2 — DIY RAG Pipeline**
